# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj). We'll use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to interact with the dataset structured under the Croissant data standard.

### Dataset Source
The dataset is available as a Croissant schema and contains several record sets and fields describing mental health survey data collected in Kilifi County, Kenya.

In [ ]:
# Ensure `mlcroissant` is available in the environment
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and records using `mlcroissant`. This provides access to the record sets, fields, and actual survey data.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets in the dataset, along with their corresponding `@id` fields, and inspect the fields (columns) of at least one record set for further exploration.

**Note:** All references to record sets, fields, and columns are via their `@id` for full reproducibility.

In [ ]:
# List all record sets in the dataset, with their names and @ids
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description if hasattr(rs, 'description') else 'No description'}\n")

# For demonstration, inspect the fields (columns) of the first record set
if record_sets:
    main_record_set = record_sets[0]
    print(f"Fields for record set '{main_record_set.name}' (@id: {main_record_set.id}):")
    for field in main_record_set.fields:
        print(f"  - {getattr(field, 'name', '')} (@id: {field.id}, dtype: {getattr(field, 'data_type', 'unknown')})")

## 3. Data Extraction
We'll now load the data records from the identified record sets. Each DataFrame will use the record set's `@id` as its key in a dictionary for easy reference. Fields (columns) will only include those documented in the record set's schema.

For this analysis, we focus on the primary record set (usually called `SurveyResponses` or similar, check its `@id` above).

In [ ]:
# Prepare list of record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Extract records for each record set
for rs_id in record_set_ids:
    print(f"Loading records from record_set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(2))
    else:
        print(f"  No records found for record set {rs_id}.")

# Choose the main record set for analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]   # You can change this if you want to focus on another
    main_df = dataframes[main_record_set_id]
    print(f"\nColumns in main DataFrame ({main_record_set_id}):\n{main_df.columns.tolist()}")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply some typical processing operations, such as filtering, normalizing, and grouping data. We demonstrate this for one of the numeric fields (e.g., GAD-7, PHQ-9, or PSQ score fields) and a grouping field (e.g., gender, age group, or education level).

**Note:** All field selections should use the field `@id` as shown in the overview step.

In [ ]:
# Select a numeric field for analysis based on field @id
# (Replace these with the actual field @ids you wish to analyze, see previous cell's printout)
numeric_field_id = None
group_field_id = None

# Try to automatically pick a numeric field (e.g., GAD-7, PHQ-9, etc.), else ask user to edit
for col in main_df.columns:
    # Heuristics: pick column with 'gad' or 'phq' or 'psq' (case-insensitive)
    if any(keyword in col.lower() for keyword in ['gad', 'phq', 'psq']):
        numeric_field_id = col
        break

# Pick a group field
for col in main_df.columns:
    if any(keyword in col.lower() for keyword in ['gender', 'sex', 'education', 'marital', 'age']):
        group_field_id = col
        break

if not numeric_field_id or not group_field_id:
    print("Could not auto-detect numeric or group field. Please set 'numeric_field_id' and 'group_field_id' manually using the field @ids from above.")
else:
    print(f"Using numeric field (by @id): {numeric_field_id}")
    print(f"Using group field (by @id): {group_field_id}")

    # Ensure numeric field is numeric
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean()

    # Filter records
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {filtered_df.shape[0]} records")
    display(filtered_df.head())

    # Normalize
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if available
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field and compare group means visually.

We use `matplotlib` and `seaborn` for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
- We loaded and explored the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library and Croissant schema.
- The dataset exposes survey results including demographic variables and multiple psychological assessment scores.
- Data was filtered, normalized, and grouped by key demographic indicators using field `@id`s for reproducibility.
- Visualizations highlight the overall distribution and group differences, aiding further statistical or social insights.

**Next steps:** Dive deeper into the survey fields, analyze trends by different subgroupings, and replicate this workflow for other Croissant-based datasets.